# 10 · Deploy the demo to Hugging Face Spaces

Uploads this folder as a Docker Space. Hugging Face builds the image and starts the app; the first build takes about 5–10 minutes.

Needs `HF_TOKEN` (write access) in the `.env` file in the repo root.

In [1]:
%pip install -q huggingface_hub python-dotenv requests

Note: you may need to restart the kernel to use updated packages.


## Settings

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv
from huggingface_hub import HfApi

HERE = Path.cwd()                       # the 10_deployment folder
load_dotenv(HERE.parent / ".env")

HF_USER = "aliissa824"
SPACE_ID = f"{HF_USER}/german-ai-text-detector"
MODEL_ID = f"{HF_USER}/german-academic-ai-text-mmbert-small"

# While the model repo is private, the Space needs a token to download it.
# Best: a separate *read-only* token as HF_READ_TOKEN in .env. Not needed once the model is public.
MODEL_IS_PUBLIC = False

api = HfApi(token=os.environ["HF_TOKEN"])
print("Logged in as:", api.whoami()["name"])

Logged in as: aliissa824


## Create the Space and upload

In [3]:
api.create_repo(SPACE_ID, repo_type="space", space_sdk="docker", exist_ok=True)

api.add_space_variable(SPACE_ID, "MODEL_ID", MODEL_ID)
if not MODEL_IS_PUBLIC:
    read_token = os.getenv("HF_READ_TOKEN")
    if not read_token:
        print("No HF_READ_TOKEN in .env, using HF_TOKEN. A read-only token is safer.")
    api.add_space_secret(SPACE_ID, "HF_TOKEN", read_token or os.environ["HF_TOKEN"])

api.upload_folder(
    repo_id=SPACE_ID, repo_type="space", folder_path=HERE,
    allow_patterns=["app/**", "Dockerfile", "requirements.txt", ".dockerignore"],
    ignore_patterns=["**/__pycache__/**"],
    commit_message="Deploy detector",
)
# The Space's README holds its settings (Docker, port, title), so it's uploaded under that name
api.upload_file(path_or_fileobj=HERE / "space_README.md", path_in_repo="README.md",
                repo_id=SPACE_ID, repo_type="space")

print(f"Space: https://huggingface.co/spaces/{SPACE_ID}")

HfHubHTTPError: Client error '402 Payment Required' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6acb87bf-41fbfa84710d2ee552a9d0c8;1ad22978-048f-47d6-a5f0-de4e947687d8)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

Static Spaces are free for everyone, but hosting Gradio and Docker Spaces on free cpu-basic requires a PRO subscription. Subscribe at https://huggingface.co/pro

## Check it

Run this once the Space shows **Running** (refresh the Space page to see the build log).

In [ ]:
import requests

url = f"https://{SPACE_ID.replace('/', '-').lower()}.hf.space"
print(requests.get(f"{url}/info", timeout=60).json())

text = ("Die vorliegende Untersuchung geht der Frage nach, wie Lehrkräfte an Grundschulen mit digitalen "
        "Lernplattformen umgehen. Grundlage sind leitfadengestützte Interviews mit zwölf Lehrkräften.")
print(requests.post(f"{url}/predict", json={"text": text}, timeout=60).json())

## Example table for the README

Sends 10 test-set texts to the running Space (a mix of classes and lengths) and prints a Markdown table to paste into the README's *Output format* section.

In [ ]:
%pip install -q datasets
from datasets import load_dataset

test = load_dataset(f"{HF_USER}/german-academic-ai-text", split="test", token=os.environ["HF_TOKEN"]).to_pandas()
NAMES = ["human", "ai_generated", "ai_assisted"]

# 10 texts: short and long, from all three classes, fixed seed so the table can be reproduced
picks = []
for label, lengths in [(0, [(0, 70), (71, 300), (301, 2000)]), (1, [(0, 70), (71, 300), (301, 2000)]),
                       (2, [(0, 70), (71, 300), (301, 600), (601, 2000)])]:
    for low, high in lengths:
        part = test[(test["label"] == label) & test["word_count"].between(low, high)]
        picks.append(part.sample(1, random_state=42).iloc[0])

print("| id | true label | words | human | ai_generated | ai_assisted | most likely | warning |")
print("|---|---|---|---|---|---|---|---|")
for row in picks:
    r = requests.post(f"{url}/predict", json={"text": row["text"]}, timeout=120).json()
    p = r["probabilities"]
    warning = "short text" if r["word_count"] < 70 else ("cut at 512 tokens" if r["truncated"] else "–")
    print(f"| {row['id']} | {NAMES[row['label']]} | {r['word_count']} | {p['human']:.1%} | {p['ai_generated']:.1%} "
          f"| {p['ai_assisted']:.1%} | {r['most_likely']} | {warning} |")